In [ ]:
import shutil
import os


source_dir = '/kaggle/input/brain-tumor-mri-dataset/Training'
dest_dir = '/kaggle/working/brain-tumor-mri-dataset/Training'


shutil.copytree(source_dir, dest_dir)

In [ ]:
source_dir_test = '/kaggle/input/brain-tumor-mri-dataset/Testing'
dest_dir_test = '/kaggle/working/brain-tumor-mri-dataset/Testing'
shutil.copytree(source_dir_test, dest_dir_test)

In [ ]:

os.listdir(dest_dir)
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import hashlib

def dhash(image, hash_size=8):
    resized = cv2.resize(image, (hash_size + 1, hash_size))
    diff = resized[:, 1:] > resized[:, :-1]
    return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])

def remove_duplicates(data_dir):
    image_hashes = {}
    duplicates = []

    for category in os.listdir(data_dir):
        category_path = os.path.join(dest_dir, category)

        for img_name in os.listdir(category_path):
            img_path = os.path.join(category_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            img_hash = dhash(img)

            if img_hash in image_hashes:
                print(f"Duplicate found: {img_name} is a duplicate of {image_hashes[img_hash]}")
                duplicates.append(img_path)
            else:
                image_hashes[img_hash] = img_name

    for duplicate in duplicates:
        os.remove(duplicate)
        print(f"Deleted duplicate image: {duplicate}")

remove_duplicates(dest_dir)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

def load_and_preprocess_images(data_dir, target_size=(256, 256), categories=['notumor', 'glioma', 'meningioma', 'pituitary']):
    images = []
    labels = []
    valid_extensions = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
    
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"The directory {data_dir} does not exist.")
    
    
    for idx, category in enumerate(categories):
        category_path = os.path.join(data_dir, category)
        if not os.path.exists(category_path):
            raise FileNotFoundError(f"The category directory {category_path} does not exist.")
        
        print(f"Processing category: {category}")
        for img_name in os.listdir(category_path):
            if any(img_name.lower().endswith(ext) for ext in valid_extensions):
                img_path = os.path.join(category_path, img_name)
                try:
                    img = load_img(img_path, color_mode='grayscale', target_size=target_size)
                    img_array = img_to_array(img)
                    images.append(img_array)
                    labels.append(idx)  # Label is the index of the category
                except Exception as e:
                    print(f"Error loading image {img_path}: {str(e)}")
    
    if not images:
        raise ValueError(f"No valid images found in the dataset.")
    
    print(f"Total images loaded: {len(images)}")
    
    
    images = np.array(images)
    labels = np.array(labels)
    
    return images, labels

def normalize_images(images):
    return (images / 127.5) - 1

def create_datasets(images, labels, batch_size=32, validation_split=0.2):
    if len(images) == 0 or len(labels) == 0:
        raise ValueError("The images or labels array is empty. Please check the data loading process.")
    
    
    labels = to_categorical(labels, num_classes=4)
    
    
    X_train, X_val, y_train, y_val = train_test_split(
        images, labels, test_size=validation_split, random_state=42, stratify=labels
    )
    
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(batch_size)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
    
    print(f"Training set size: {len(X_train)}")
    print(f"Validation set size: {len(X_val)}")
    
    return train_dataset, val_dataset


data_dir = '/kaggle/working/brain-tumor-mri-dataset/Training'
categories = ['notumor', 'glioma', 'meningioma', 'pituitary']


if __name__ == "__main__":
    images, labels = load_and_preprocess_images(data_dir, categories=categories)
    normalized_images = normalize_images(images)
    
    
    train_dataset, val_dataset = create_datasets(normalized_images, labels)
    print("Datasets created successfully.")


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def create_resnet_model(input_shape=(256, 256, 1), num_classes=4):
    # Load the ResNet50 model, pre-trained on ImageNet and excluding the top layers
    base_model = ResNet50(weights=None, include_top=False, input_shape=input_shape)

    
    model = models.Sequential()
    model.add(base_model)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model


input_shape = (256, 256, 1)
num_classes = 4
model = create_resnet_model(input_shape=input_shape, num_classes=num_classes)


early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, min_lr=1e-6)


history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, reduce_lr]
)


In [ ]:

test_data_dir = '/kaggle/working/brain-tumor-mri-dataset/Testing'  # Change this to your actual test directory path

test_images, test_labels = load_and_preprocess_images(test_data_dir, categories=categories)
normalized_test_images = normalize_images(test_images)


test_labels_categorical = to_categorical(test_labels, num_classes=4)
test_dataset = tf.data.Dataset.from_tensor_slices((normalized_test_images, test_labels_categorical)).batch(32)

print(f"Test set size: {len(normalized_test_images)}")


In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns


test_predictions = model.predict(test_dataset)
test_pred_labels = np.argmax(test_predictions, axis=1)
true_test_labels = np.argmax(test_labels_categorical, axis=1)

print(classification_report(true_test_labels, test_pred_labels, target_names=categories))

cm = confusion_matrix(true_test_labels, test_pred_labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix on Test Data')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()


In [ ]:
def visualize_predictions_on_test(model, test_dataset, num_images=5):
    test_images, test_labels = next(iter(test_dataset))
    predictions = model.predict(test_images)

    for i in range(num_images):
        plt.figure(figsize=(5, 5))
        plt.imshow(test_images[i].numpy().squeeze(), cmap='gray')
        pred_label = np.argmax(predictions[i])
        true_label = np.argmax(test_labels[i])
        plt.title(f'Pred: {categories[pred_label]}, True: {categories[true_label]}')
        plt.axis('off')
        plt.show()

visualize_predictions_on_test(model, test_dataset)


In [ ]:
def plot_training_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 6))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b', label='Training accuracy')
    plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b', label='Training loss')
    plt.plot(epochs, val_loss, 'r', label='Validation loss')
    plt.title('Training and validation loss')
    plt.legend()

    plt.show()


plot_training_history(history)


In [ ]:
model.save('/kaggle/working/resnet_brain_tumor_model.h5')

